In [ ]:
# ==============================
# STEP 1: Install Libraries
# ==============================

!pip install fastapi uvicorn pyngrok scikit-learn joblib nest-asyncio -q


# ==============================
# STEP 2: Import Libraries
# ==============================

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib
import numpy as np

from fastapi import FastAPI
from pydantic import BaseModel

import uvicorn
import nest_asyncio
from threading import Thread

from pyngrok import ngrok
import requests


# ==============================
# STEP 3: Train Model
# ==============================

iris = load_iris()

X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

model = RandomForestClassifier()

model.fit(
    X_train,
    y_train
)

pred = model.predict(X_test)

print("\nModel Accuracy:")
print(
    accuracy_score(
        y_test,
        pred
    )
)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        pred
    )
)

# Save model

joblib.dump(
    model,
    "model.pkl"
)

print("\nModel Saved Successfully")


# ==============================
# STEP 4: Build FastAPI
# ==============================

app = FastAPI()

saved_model = joblib.load(
    "model.pkl"
)

class InputData(BaseModel):
    features: list


@app.get("/")
def home():

    return {
        "message":
        "API Running Successfully"
    }


@app.post("/predict")
def predict(data: InputData):

    arr = np.array(
        data.features
    ).reshape(1,-1)

    prediction = saved_model.predict(arr)

    species = iris.target_names[
        prediction[0]
    ]

    return {

        "prediction":
        int(prediction[0]),

        "species":
        str(species)

    }


# ==============================
# STEP 5: Start Server
# ==============================

nest_asyncio.apply()

def run():

    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000
    )


thread = Thread(
    target=run
)

thread.start()


# ==============================
# STEP 6: Add ngrok token
# ==============================

# Replace with YOUR ngrok token

NGROK_TOKEN = "3D7xoUVJ9goursklkmWJEFX5GiM_2FwDr4RUCaQ3kzPmRk7eT"

ngrok.set_auth_token(
    NGROK_TOKEN
)


public_url = ngrok.connect(
    8000
)

url = public_url.public_url

print("\nPublic API URL:")
print(url)


# ==============================
# STEP 7: Test API
# ==============================

sample = {

    "features":
    [5.1,3.5,1.4,0.2]

}

response = requests.post(

    url + "/predict",

    json=sample

)

print("\nPrediction Result:")

print(
    response.json()
)


print("\nDeployment Successful")

"""
{
  "features":[5.1,3.5,1.4,0.2]
}
"""